
Week -6 : with Neural network terminology

In [1]:

#Import Section
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

from scipy.stats import norm
from scipy.stats import qmc

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# Creating training tensors
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
def training_tensors(X, y):
    X_tensor = torch.from_numpy(X).float()

    y_class = (y >= np.percentile(y, 80)).astype(float)
    y_tensor = torch.from_numpy(y_class).float().reshape(len(X),1)

    return X_tensor, y_tensor, y_class


# Function for training the neural network iteratively
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
def train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser):
    
    loss_history = []
    accuracy_history = []

    for epoch in range(epochs):
        model.train()

        # Forward pass
        y_pred_nn = model(X_tensor)
        loss = criterion(y_pred_nn, y_tensor)

        # Tracking loss
        loss_history.append(loss.item())

        # Backward pass
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        # Tracking accuracy
        y_pred_nn_np = y_pred_nn.detach().numpy()
        y_pred_nn_class = (y_pred_nn_np >= np.percentile(y_pred_nn_np, 80)).astype(float)
        
        accuracy = accuracy_score(y_class, y_pred_nn_class)
        accuracy_history.append(accuracy)

        # Printing epochs
        if epoch % 10 == 0:
            print(f"Epoch: {epoch}, Loss: {loss}, Accuracy: {accuracy}")
    
    model.eval()

    return model, y_pred_nn, loss_history, accuracy_history

Function -1 [2-Dimensional]  Load Initial Data
The initial dataset consists of:

inputs: 2‑D spatial coordinates

outputs: corresponding sensor readings

In [3]:
function1_inputs = np.load(
    '../data/week6/function_1/inputs.npy')
print("Inputs:\n", function1_inputs)

function1_outputs = np.load(
    '../data/week6/function_1/outputs.npy')
print("Outputs:\n", function1_outputs)

print("Output type:", type(function1_outputs))

# Assign to working variables
X = function1_inputs
y = function1_outputs

Inputs:
 [[0.31940389 0.76295937]
 [0.57432921 0.8798981 ]
 [0.73102363 0.73299988]
 [0.84035342 0.26473161]
 [0.65011406 0.68152635]
 [0.41043714 0.1475543 ]
 [0.31269116 0.07872278]
 [0.68341817 0.86105746]
 [0.08250725 0.40348751]
 [0.88388983 0.58225397]
 [0.65       0.3       ]
 [0.15       0.85      ]
 [0.85       0.15      ]
 [0.2        0.8       ]
 [0.85       0.45      ]]
Outputs:
 [ 1.32267704e-079  1.03307824e-046  7.71087511e-016  3.34177101e-124
 -3.60606264e-003 -2.15924904e-054 -2.08909327e-091  2.53500115e-040
  3.60677119e-081  6.22985647e-048 -3.98693469e-048  2.85566405e-181
  2.85566405e-181  1.59622584e-134 -6.50000083e-057]
Output type: <class 'numpy.ndarray'>


Neural Network using nn.Sequential for Function -1

In [4]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 100

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
X_candidates = np.random.uniform(0, 1, (50000, 2))

with torch.no_grad(): # Disable gradients for faster computation
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

Epoch: 0, Loss: 0.7248929738998413, Accuracy: 0.8666666666666667
Epoch: 10, Loss: 0.5065250396728516, Accuracy: 0.6
Epoch: 20, Loss: 0.4595692455768585, Accuracy: 0.8666666666666667
Epoch: 30, Loss: 0.3823334276676178, Accuracy: 1.0
Epoch: 40, Loss: 0.2832576632499695, Accuracy: 1.0
Epoch: 50, Loss: 0.2054329365491867, Accuracy: 1.0
Epoch: 60, Loss: 0.16916264593601227, Accuracy: 1.0
Epoch: 70, Loss: 0.1492622345685959, Accuracy: 1.0
Epoch: 80, Loss: 0.13161319494247437, Accuracy: 1.0
Epoch: 90, Loss: 0.1136888936161995, Accuracy: 1.0


Apply GaussianProcessRegression

In [5]:
def expected_improvement(mu, sigma, y_best):
    improvement = (mu - y_best)
    Z = improvement / (sigma + 1e-9)

    EI = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    EI[sigma == 0] = 0
    
    return EI

In [6]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * RBF(length_scale=1)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=12)

model = gaussianP.fit(X, y)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# Acquisition function (UCB)
k = 0.15

best_value1 = np.max(y)
ei = expected_improvement(mu, sigma, best_value1)

acquisition_function = ei * (0.5 + 0.5 * X_candidates_prob)

# Pick next point
x_next = X_candidates[np.argmax(acquisition_function)]

print('The next query point is:', np.round(x_next, 6))

The next query point is: [0.857818 0.784196]


C:\Users\ruchi\anaconda3\envs\mlenv\Lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Function -2 [2-Dimensional]

In [7]:
function2_inputs = np.load(
    '../data/week6/function_2/inputs.npy')
print("Inputs:\n", function2_inputs)

function2_outputs = np.load(
    '../data/week6/function_2/outputs.npy')
print("Outputs:\n", function2_outputs)

print("Output type:", type(function2_outputs))

# Assign to working variables
X = function2_inputs
y = function2_outputs

y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

Inputs:
 [[0.66579958 0.12396913]
 [0.87779099 0.7786275 ]
 [0.14269907 0.34900513]
 [0.84527543 0.71112027]
 [0.45464714 0.29045518]
 [0.57771284 0.77197318]
 [0.43816606 0.68501826]
 [0.34174959 0.02869772]
 [0.33864816 0.21386725]
 [0.70263656 0.9265642 ]
 [0.2        0.4       ]
 [0.35       0.65      ]
 [0.4        0.7       ]
 [0.5        0.8       ]
 [0.55       0.65      ]]
Outputs:
 [ 0.53899612  0.42058624 -0.06562362  0.29399291  0.21496451  0.02310555
  0.24461934  0.03874902 -0.01385762  0.61120522 -0.0694397   0.09282575
  0.03634587  0.49795941  0.27219175]
Output type: <class 'numpy.ndarray'>


In [8]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 80

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
X_candidates = np.random.uniform(0, 1, (50000, 2))

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

Epoch: 0, Loss: 0.7106910347938538, Accuracy: 0.8666666666666667
Epoch: 10, Loss: 0.5298307538032532, Accuracy: 0.7333333333333333
Epoch: 20, Loss: 0.4898432791233063, Accuracy: 0.7333333333333333
Epoch: 30, Loss: 0.4610215127468109, Accuracy: 0.7333333333333333
Epoch: 40, Loss: 0.424447625875473, Accuracy: 0.7333333333333333
Epoch: 50, Loss: 0.386391282081604, Accuracy: 0.7333333333333333
Epoch: 60, Loss: 0.35190534591674805, Accuracy: 0.8666666666666667
Epoch: 70, Loss: 0.3154750168323517, Accuracy: 0.8666666666666667


In [9]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * RBF(length_scale=1)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=15)

model = gaussianP.fit(X, y_scaled)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# Acquisition function (UCB)
k = 1
ucb = mu + k * sigma
acquisition_function = ucb * (0.5 + 0.5 * X_candidates_prob)


best_value2_scaled = np.max(y_scaled)
ei = expected_improvement(mu, sigma, best_value2_scaled)
acquisition = ei * (0.5 + 0.5 * X_candidates_prob)

# Pick next point
x_next = X_candidates[np.argmax(acquisition_function)]

print('The next query point is:', np.round(x_next, 6))

The next query point is: [0.712994 0.951759]


Function -3 [3-Dimensional] Start with Balanced k i.e. between 1-2 and RBF kernel

In [10]:
function3_inputs = np.load(
    '../data/week6/function_3/inputs.npy')
print("Inputs:\n", function3_inputs)

function3_outputs = np.load(
    '../data/week6/function_3/outputs.npy')
print("Outputs:\n", function3_outputs)

print("Output type:", type(function3_outputs))

# Assign to working variables
X = function3_inputs
y = function3_outputs

y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

Inputs:
 [[0.17152521 0.34391687 0.2487372 ]
 [0.24211446 0.64407427 0.27243281]
 [0.53490572 0.39850092 0.17338873]
 [0.49258141 0.61159319 0.34017639]
 [0.13462167 0.21991724 0.45820622]
 [0.34552327 0.94135983 0.26936348]
 [0.15183663 0.43999062 0.99088187]
 [0.64550284 0.39714294 0.91977134]
 [0.74691195 0.28419631 0.22629985]
 [0.17047699 0.6970324  0.14916943]
 [0.22054934 0.29782524 0.34355534]
 [0.66601366 0.67198515 0.2462953 ]
 [0.04680895 0.23136024 0.77061759]
 [0.60009728 0.72513573 0.06608864]
 [0.96599485 0.86111969 0.56682913]
 [0.7        0.25       0.6       ]
 [0.72       0.32       0.55      ]
 [0.74       0.3        0.58      ]
 [0.3        0.7        0.4       ]
 [0.2        0.8        0.3       ]]
Outputs:
 [-0.1121222  -0.08796286 -0.11141465 -0.03483531 -0.04800758 -0.11062091
 -0.39892551 -0.11386851 -0.13146061 -0.09418956 -0.04694741 -0.10596504
 -0.11804826 -0.03637783 -0.05675837 -0.0940786  -0.03703186 -0.05640717
 -0.03778947 -0.0848159 ]
Output type: <c

In [11]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 100

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
X_candidates = np.random.uniform(0, 1, (30000, 3))

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

Epoch: 0, Loss: 0.6599355936050415, Accuracy: 0.6
Epoch: 10, Loss: 0.49290961027145386, Accuracy: 0.7
Epoch: 20, Loss: 0.4608851969242096, Accuracy: 0.7
Epoch: 30, Loss: 0.435666024684906, Accuracy: 0.8
Epoch: 40, Loss: 0.4157995283603668, Accuracy: 0.8
Epoch: 50, Loss: 0.3937308192253113, Accuracy: 0.8
Epoch: 60, Loss: 0.35911256074905396, Accuracy: 0.8
Epoch: 70, Loss: 0.3178548812866211, Accuracy: 0.8
Epoch: 80, Loss: 0.27691587805747986, Accuracy: 0.9
Epoch: 90, Loss: 0.24455127120018005, Accuracy: 0.9


In [12]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * RBF(length_scale=1)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)

model = gaussianP.fit(X, y_scaled)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# Acquisition function (UCB)
k = 1.0
acquisition_function = (mu + k * sigma) * (0.5 + 0.5 * X_candidates_prob)

# Pick next point
x_next = X_candidates[np.argmax(acquisition_function)]

print('The next query point is:', np.round(x_next, 6))

The next query point is: [0.513377 0.524226 0.547004]


Function-4  [4-diemnsional] For many‑maxima, non‑smooth functions, use a Matern(ν = 1.5 or 2.5) kernel, high κ with slow decay, and inject explicit random exploration. This combination prevents early over‑commitment and keeps the search globally aware.

In [13]:
function4_inputs = np.load(
    '../data/week6/function_4/inputs.npy')
print("Inputs:\n", function4_inputs)

function4_outputs = np.load(
    '../data/week6/function_4/outputs.npy')
print("Outputs:\n", function4_outputs)

print("Output type:", type(function4_outputs))

# Assign to working variables
X = function4_inputs
y = function4_outputs

y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

Inputs:
 [[0.89698105 0.72562797 0.17540431 0.70169437]
 [0.8893564  0.49958786 0.53926886 0.50878344]
 [0.25094624 0.03369313 0.14538002 0.49493242]
 [0.34696206 0.0062504  0.76056361 0.61302356]
 [0.12487118 0.12977019 0.38440048 0.2870761 ]
 [0.80130271 0.50023109 0.70664456 0.19510284]
 [0.24770826 0.06044543 0.04218635 0.44132425]
 [0.74670224 0.7570915  0.36935306 0.20656628]
 [0.40066503 0.07257425 0.88676825 0.24384229]
 [0.6260706  0.58675126 0.43880578 0.77885769]
 [0.95713529 0.59764438 0.76611385 0.77620991]
 [0.73281243 0.14524998 0.47681272 0.13336573]
 [0.65511548 0.07239183 0.68715175 0.08151656]
 [0.21973443 0.83203134 0.48286416 0.08256923]
 [0.48859419 0.2119651  0.93917791 0.37619173]
 [0.16713049 0.87655456 0.21723954 0.95980098]
 [0.21691119 0.16608583 0.24137226 0.77006248]
 [0.38748784 0.80453226 0.75179548 0.72382744]
 [0.98562189 0.66693268 0.15678328 0.8565348 ]
 [0.03782483 0.66485335 0.16198218 0.25392378]
 [0.68348638 0.9027701  0.33541983 0.99948256]
 [0.

In [14]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 40

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
X_candidates = np.random.uniform(0, 1, (50000, 4))

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

Epoch: 0, Loss: 0.6581229567527771, Accuracy: 0.6
Epoch: 10, Loss: 0.41584572196006775, Accuracy: 0.8285714285714286
Epoch: 20, Loss: 0.31002339720726013, Accuracy: 0.8857142857142857
Epoch: 30, Loss: 0.2255433201789856, Accuracy: 0.8857142857142857


In [15]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3)) \
         * Matern(length_scale=1,
                  length_scale_bounds=(1e-6, 1e3),
                  nu=2.5)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)

model = gaussianP.fit(X, y_scaled)

#sampler = qmc.Sobol(4) # Using smarter sampling
#X_candidates = sampler.random(2**15)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# Acquisition function (UCB)
k = 1.5
acquisition_function = (mu + k * sigma) * (0.5 + 0.5 * X_candidates_prob)

# Pick next point
x_next = X_candidates[np.argmax(acquisition_function)]

print('The next query point is:', np.round(x_next, 6))

The next query point is: [0.300542 0.426256 0.480953 0.360808]


Function-5  [4-diemnsional] 

In [16]:
function5_inputs = np.load(
    '../data/week6/function_5/inputs.npy')
print("Inputs:\n", function5_inputs)

function5_outputs = np.load(
    '../data/week6/function_5/outputs.npy')
print("Outputs:\n", function5_outputs)

print("Output type:", type(function5_outputs))

# Assign to working variables
X = function5_inputs
y = function5_outputs

y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.81       0.6        0.2        0.412     ]
 [0.

In [17]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 20

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
X_candidates = np.random.uniform(0, 1, (50000, 4))

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

Epoch: 0, Loss: 0.7338747978210449, Accuracy: 0.6
Epoch: 10, Loss: 0.4057265520095825, Accuracy: 0.92


In [18]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * RBF(length_scale=1)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=12)

model = gaussianP.fit(X, y_scaled)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

y_func5_scaled = np.max(y_scaled)
ei = expected_improvement(mu, sigma, y_func5_scaled)
acquisition_function = ei * (0.5 + 0.5 * X_candidates_prob)


# Pick next point
x_next = X_candidates[np.argmax(acquisition_function)]

print('The next query point is:', np.round(x_next, 6))

The next query point is: [0.30868  0.809317 0.967206 0.931763]


Function -6 [5-Dimensional]

In [19]:
function6_inputs = np.load(
    '../data/week6/function_6/inputs.npy')
print("Inputs:\n", function6_inputs)

function6_outputs = np.load(
    '../data/week6/function_6/outputs.npy')
print("Outputs:\n", function6_outputs)

print("Output type:", type(function6_outputs))

# Assign to working variables
X = function6_inputs
y = function6_outputs
y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

Inputs:
 [[0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]
 [0.24238435 0.84409997 0.5778091  0.67902128 0.50195289]
 [0.72952261 0.7481062  0.67977464 0.35655228 0.67105368]
 [0.77062024 0.11440374 0.04677993 0.64832428 0.27354905]
 [0.6188123  0.33180214 0.18728787 0.75623847 0.3288348 ]
 [0.78495809 0.91068235 0.7081201  0.95922543 0.0049115 ]
 [0.14511079 0.8966846  0.89632223 0.72627154 0.23627199]
 [0.94506907 0.28845905 0.97880576 0.96165559 0.59801594]
 [0.12572016 0.86272469 0.02854433 0.24660527 0.75120624]
 [0.75759436 0.35583141 0.0165229  0.4342072  0.11243304]
 [0.5367969  0.30878091 0.41187929 0.38822518 0.5225283 ]
 [0.95773967 0.23566857 0.09914585 0.15680593 0.07131737]
 [0.6293079  0.80348368 0.81140844 0.04561319 0.11062446]
 [0.02173531 0.42808424 0.83593944 0.48948866 0.51108173]
 [0.43934426 0.69892383 0.42682022 0.10947609 0.87788847]
 [0.25890557 0.79367771 0.6421139  0.19667346 0.59310318]
 [0.43216593 0.71561781 0.3418191  0.70499988 0.61496184]
 [0.7

In [20]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 30

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
X_candidates = np.random.uniform(0, 1, (50000, 5))

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

Epoch: 0, Loss: 0.6266019940376282, Accuracy: 0.76
Epoch: 10, Loss: 0.4593941867351532, Accuracy: 0.84
Epoch: 20, Loss: 0.3624351918697357, Accuracy: 0.84


In [21]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * RBF(length_scale=1)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)

model = gaussianP.fit(X, y_scaled)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# Acquisition function (UCB)
k = 1.5
acquisition_function = (mu + k * sigma) * (0.5 + 0.5 * X_candidates_prob)

# Pick next point
x_next = X_candidates[np.argmax(acquisition_function)]

print('The next query point is:', np.round(x_next, 6))

The next query point is: [0.327981 0.318931 0.54671  0.816449 0.084858]


In [ ]:
Function -7 [6-Dimensional]

In [22]:
function7_inputs = np.load(
    '../data/week6/function_7/inputs.npy')
print("Inputs:\n", function7_inputs)

function7_outputs = np.load(
    '../data/week6/function_7/outputs.npy')
print("Outputs:\n", function7_outputs)

print("Output type:", type(function7_outputs))

# Assign to working variables
X = function7_inputs
y = function7_outputs

y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

Inputs:
 [[0.27262382 0.32449536 0.89710881 0.83295115 0.15406269 0.79586362]
 [0.54300258 0.9246939  0.34156746 0.64648585 0.71844033 0.34313266]
 [0.09083225 0.66152938 0.06593091 0.25857701 0.96345285 0.6402654 ]
 [0.11886697 0.61505494 0.90581639 0.8553003  0.41363143 0.58523563]
 [0.63021764 0.8380969  0.68001305 0.73189509 0.52673671 0.34842921]
 [0.76491917 0.25588292 0.60908422 0.21807904 0.32294277 0.09579366]
 [0.05789554 0.49167222 0.24742222 0.21811844 0.42042833 0.73096984]
 [0.19525188 0.07922665 0.55458046 0.17056682 0.01494418 0.10703171]
 [0.64230298 0.83687455 0.02179269 0.10148801 0.68307083 0.6924164 ]
 [0.78994255 0.19554501 0.57562333 0.07365919 0.25904917 0.05109986]
 [0.52849733 0.45742436 0.36009569 0.36204551 0.81689098 0.63747637]
 [0.72261522 0.01181284 0.06364591 0.16517311 0.07924415 0.35995166]
 [0.07566492 0.33450212 0.13273274 0.60831236 0.91838592 0.82233079]
 [0.94245084 0.37743962 0.48612233 0.22879108 0.08263175 0.71195755]
 [0.14864702 0.03394336 0

In [23]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 15

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
X_candidates = np.random.uniform(0, 1, (50000, 6))

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

Epoch: 0, Loss: 0.7439205050468445, Accuracy: 0.6
Epoch: 10, Loss: 0.4299792945384979, Accuracy: 0.8857142857142857


In [24]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3)) \
         * Matern(length_scale=1,
                  length_scale_bounds=(1e-6, 1e3),
                  nu=2.5)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)

model = gaussianP.fit(X, y_scaled)

#sampler = qmc.Sobol(4) # Using smarter sampling
#X_candidates = sampler.random(2**15)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# Acquisition function (UCB)
k = 1.0
acquisition_function = (mu + k * sigma) * (0.5 + 0.5 * X_candidates_prob)

# Pick next point
x_next = X_candidates[np.argmax(acquisition_function)]

print('The next query point is:', np.round(x_next, 6))

The next query point is: [0.089807 0.448014 0.293217 0.241971 0.332065 0.756019]


Function -8 [8-Dimensional]

In [25]:
function8_inputs = np.load(
    '../data/week6/function_8/inputs.npy')
print("Inputs:\n", function8_inputs)

function8_outputs = np.load(
    '../data/week6/function_8/outputs.npy')
print("Outputs:\n", function8_outputs)

print("Output type:", type(function8_outputs))

# Assign to working variables
X = function8_inputs
y = function8_outputs

y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

Inputs:
 [[0.60499445 0.29221502 0.90845275 0.35550624 0.20166872 0.57533801
  0.31031095 0.73428138]
 [0.17800696 0.56622265 0.99486184 0.21032501 0.32015266 0.70790879
  0.63538449 0.10713163]
 [0.00907698 0.81162615 0.52052036 0.07568668 0.26511183 0.09165169
  0.59241515 0.36732026]
 [0.50602816 0.65373012 0.36341078 0.17798105 0.0937283  0.19742533
  0.7558269  0.29247234]
 [0.35990926 0.24907568 0.49599717 0.70921498 0.11498719 0.28920692
  0.55729515 0.59388173]
 [0.77881834 0.0034195  0.33798313 0.51952778 0.82090699 0.53724669
  0.5513471  0.66003209]
 [0.90864932 0.0622497  0.23825955 0.76660355 0.13233596 0.99024381
  0.68806782 0.74249594]
 [0.58637144 0.88073573 0.74502075 0.54603485 0.00964888 0.74899176
  0.23090707 0.09791562]
 [0.76113733 0.85467239 0.38212433 0.33735198 0.68970832 0.30985305
  0.63137968 0.04195607]
 [0.9849332  0.69950626 0.9988855  0.18014846 0.58014315 0.23108719
  0.49082694 0.31368272]
 [0.11207131 0.43773566 0.59659878 0.59277563 0.22698177 0.41

In [26]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 20

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
sampler = qmc.Sobol(8) # Using smarter sampling
X_candidates = sampler.random(2**15)

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

Epoch: 0, Loss: 0.6686577796936035, Accuracy: 0.6888888888888889
Epoch: 10, Loss: 0.37627583742141724, Accuracy: 0.8666666666666667


In [31]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * Matern(length_scale=1, nu=2.5)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)

model = gaussianP.fit(X, y_scaled)

#sampler = qmc.Sobol(4) # Using smarter sampling for 4-diemnsions
#X_candidates = sampler.random(2**15)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# Acquisition function (UCB)
k = 1
acquisition_function = (mu + k * sigma) * (0.5 + 0.5 * X_candidates_prob)

# Pick next point
x_next = X_candidates[np.argmax(acquisition_function)]

print('The next query point is:', np.round(x_next, 6))

C:\Users\ruchi\anaconda3\envs\mlenv\Lib\site-packages\sklearn\gaussian_process\_gpr.py:670: ConvergenceWarning: lbfgs failed to converge after 10 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


The next query point is: [0.053026 0.084463 0.039813 0.08382  0.985288 0.574184 0.069354 0.722959]


C:\Users\ruchi\anaconda3\envs\mlenv\Lib\site-packages\sklearn\gaussian_process\_gpr.py:670: ConvergenceWarning: lbfgs failed to converge after 7 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
